# ERA5 Preprocessing and Regridding to N320

## 1. Purpose

This notebook preprocesses ERA5 data by:

- **Loading** surface, pressure-level, and static variables  
- **Selecting** the final two timesteps for each file  
- **Regridding** all variables to the N320 Gaussian grid  
- **Saving** the processed data as serialized `.pkl` files  

### Why this is necessary

ERA5 data is provided on a regular latitude–longitude grid (0.25° resolution), while many atmospheric models use Gaussian grids such as **N320**.

Regridding ensures:

- consistent spatial representation  
- compatibility with downstream models  
- reduced preprocessing during training  

## 2. Imports

This cell imports all required libraries.

- **numpy**: numerical operations  
- **earthkit.regrid**: grid interpolation  
- **xarray**: handling NetCDF data  
- **pathlib**: file paths  
- **glob**: file matching  
- **datetime**: time iteration  
- **pickle**: saving outputs  

In [1]:
print("FMAifs PREPROCESS CPU REGRID")

import numpy as np
import earthkit.regrid as ekr
from pathlib import Path
import glob
from datetime import datetime, timedelta
import xarray as xr
import pickle
import fsspec

FMAifs PREPROCESS CPU REGRID


## 3. Regridding Function

This function:

- takes a **2D field** (latitude × longitude)  
- converts it from ERA5 grid (0.25°) to N320 Gaussian grid  

This ensures all variables share the same spatial grid.

In [2]:
def regrid_to_n320(arr_2d):
    return ekr.interpolate(
        arr_2d,
        {"grid": (0.25, 0.25)},
        {"grid": "N320"}
    )

In [3]:
#data_path = Path("../era5")
data_path = "s3://enw-04241552-kx1nks-shared/data/era5"
output_path = Path("./era5/aifs_preprocessed")
#output_path.mkdir(exist_ok=True)
output_path.mkdir(parents=True, exist_ok=True)

PARAM_SFC = ["10u", "10v", "2d", "2t", "msl", "skt", "sp", "tcw"]
PARAM_STATIC = ["lsm", "z", "slor", "sdor"]
PARAM_PL = ["z", "t", "u", "v", "w", "q"]

rename_sfc = {
    "u10": "10u",
    "v10": "10v",
    "d2m": "2d",
    "t2m": "2t"
}

## 4. Paths and Variables

- `data_path`: location of ERA5 input files  
- `output_path`: directory for processed outputs  

### Variable groups

- **Surface variables**: near-ground atmospheric conditions  
- **Static variables**: time-invariant fields  
- **Pressure-level variables**: vertical atmospheric structure  

The `rename_sfc` dictionary standardizes naming.

## 5. Static Dataset

Static variables:

- do **not change over time**  
- are loaded **once**  
- are reused for every timestep  

In [4]:
#static_ds = xr.open_dataset(data_path / "static" / "era5_static-allvar.nc")
static_ds = xr.open_dataset(
    f"{data_path}/static/era5_static-allvar.nc",
    engine="h5netcdf",
)

## 6. Time Range

- Defines the processing window  
- Iterates in **6-hour increments**  

In [5]:
start_date = datetime(2024, 5, 1, 0)
end_date   = datetime(2024, 5, 2, 0)

current_date = start_date

## 7. Main Loop

Each iteration:

- processes one timestamp  
- builds file paths  
- prepares output naming

## 8. Load Data

- Loads multiple NetCDF files  
- Concatenates along the **time dimension**  
- Renames surface variables for consistency

## 9. Select Timesteps

Only the **last two timesteps** are used.

This is useful for:

- short temporal sequences  
- model input/output pairs

## 10. Output Container

A dictionary is used to store all processed variables.

## 11. Static Variables

Steps:

- **squeeze** removes extra dimensions  
- **roll** aligns longitude  
- **regrid** converts to N320  
- **reshape** flattens spatial grid  
- **concatenate** duplicates across time  

Static fields are duplicated because they do not change with time.

## 12. Surface Variables

- Each timestep is regridded independently  
- Results are stacked into a 2-step array
- 
## 13. Pressure-Level Variables

- Data includes a **vertical dimension**  
- Each level is processed separately  

Output naming example:

- `t_500`  
- `u_850`  

This converts 3D data into multiple 2D fields.

## 14. Saving Output

Each file contains:

- **date**  
- **fields dictionary**  

Saved in `.pkl` format for efficient reuse.

## 15. Time Advancement

- Advances by **6 hours per loop iteration**  
- Matches ERA5 temporal resolution  

In [6]:
fs = fsspec.filesystem("s3", 
                       use_listings_cache=False,)

while current_date <= end_date:

    print(f"\nProcessing {current_date}")

    yyyy = str(current_date.year)
    mm = f"{current_date.month:02d}"
    dd = f"{current_date.day:02d}"
    
    timestamp = current_date.strftime("%Y%m%d_%H")
    
    pattern = (
        f"{data_path}/surface_hourly/inst/"
        f"Y{yyyy}/M{mm}/"
        f"era5_surface-inst_allvar_{yyyy}{mm}{dd}_*.nc"
    )
    surf_files = sorted(fs.glob(pattern))
    files = [f if f.startswith("s3://") else f"s3://{f}" for f in surf_files]
    print(files)
    if not files:
        raise FileNotFoundError(f"No files found for pattern: {pattern}")
    surf = xr.open_mfdataset(
        files,
        engine="h5netcdf",
        combine="nested",
        concat_dim="valid_time",
    ).rename(rename_sfc)

    pattern = (
        f"{data_path}/pressure_hourly/inst/"
        f"Y{yyyy}/M{mm}/"
        f"era5_atmos-inst_allvar_{yyyy}{mm}{dd}_*.nc"
    )
    atmos_files = sorted(fs.glob(pattern))
    files = [f if f.startswith("s3://") else f"s3://{f}" for f in atmos_files]
    if not files:
        raise FileNotFoundError(f"No files found for pattern: {pattern}")
    atmos = xr.open_mfdataset(
        files,
        engine="h5netcdf",
        combine="nested",
        concat_dim="valid_time",
    )
    surf = surf.isel(valid_time=[-2, -1])
    atmos = atmos.isel(valid_time=[-2, -1])

    fields = {}

    # --- STATIC ---
    for var in PARAM_STATIC:
        data = np.squeeze(static_ds[var].values)
        data = np.roll(data, -data.shape[1] // 2, axis=1)
        vals = regrid_to_n320(data)
        flat = vals.reshape(1, -1)
        fields[var] = np.concatenate([flat, flat], axis=0)

    # --- SURFACE ---
    for var in PARAM_SFC:
        data = surf[var].values
        fields[var] = np.stack([
            regrid_to_n320(data[t]) for t in range(2)
        ])

    # --- PRESSURE LEVELS ---
    levels = atmos["pressure_level"].values

    for var in PARAM_PL:
        data = atmos[var].values

        for i, level in enumerate(levels):
            slice_ = data[:, i]

            out = np.stack([
                regrid_to_n320(slice_[t]) for t in range(2)
            ])

            name = f"{var}_{int(level)}"
            fields[name] = out

    # save
    out_file = output_path / f"{timestamp}.pkl"

    with open(out_file, "wb") as f:
        pickle.dump({
            "date": current_date,
            "fields": fields
        }, f)

    print("Saved:", out_file)

    current_date += timedelta(hours=6)



Processing 2024-05-01 00:00:00
['s3://enw-04241552-kx1nks-shared/data/era5/surface_hourly/inst/Y2024/M05/era5_surface-inst_allvar_20240501_00z.nc', 's3://enw-04241552-kx1nks-shared/data/era5/surface_hourly/inst/Y2024/M05/era5_surface-inst_allvar_20240501_06z.nc', 's3://enw-04241552-kx1nks-shared/data/era5/surface_hourly/inst/Y2024/M05/era5_surface-inst_allvar_20240501_12z.nc', 's3://enw-04241552-kx1nks-shared/data/era5/surface_hourly/inst/Y2024/M05/era5_surface-inst_allvar_20240501_18z.nc']
Saved: era5/aifs_preprocessed/20240501_00.pkl

Processing 2024-05-01 06:00:00
['s3://enw-04241552-kx1nks-shared/data/era5/surface_hourly/inst/Y2024/M05/era5_surface-inst_allvar_20240501_00z.nc', 's3://enw-04241552-kx1nks-shared/data/era5/surface_hourly/inst/Y2024/M05/era5_surface-inst_allvar_20240501_06z.nc', 's3://enw-04241552-kx1nks-shared/data/era5/surface_hourly/inst/Y2024/M05/era5_surface-inst_allvar_20240501_12z.nc', 's3://enw-04241552-kx1nks-shared/data/era5/surface_hourly/inst/Y2024/M05/era